<a href="https://colab.research.google.com/github/MatteoOnger/lama-lab/blob/main/notebooks/market_making.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Market Making Pipeline & Analysis

Run market making experiments, load saved artifacts, and interactively inspect execution results.

In [ ]:
# Do NOT run this cell in local environment - it's intended for Google Colab only.

# Clone GitHub repository
!git clone https://github.com/MatteoOnger/lama-lab.git

# Set working directory
%cd /content/lama-lab

# Install dependencies
%pip install -q -e .

# Set working directory
%cd /content/lama-lab/notebooks

## Run Experiment

Configure parameters and execute the simulation pipeline.

In [ ]:
# Add project root to sys.path if running from inside the notebooks/ directory
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
from scripts.market_making import run_pipeline

# Custom experiment configuration
config = {
    "experiment_name": "pzomd_1fp",
    "n_samples": 1000000,
    "history_window": 1,
    "env": {
        "n_makers": 2,
        "n_episodes": 100000,
        "n_rounds": 100000,
        "epsilon": 0.001,
    },
    "generator": {
        "_target_": "lama_lab.generators.GaussianMixtureGenerator",
        "weights": [1.0],
        "means": [0.5],
        "stds": [0.1],
        "low": 0.0,
        "high": 1.0,
    },
    "agent": {
        "_target_": "lama_lab.agents.AgentPZOMD",
        "init_x": [0.0, 1.0],
        "delta_0": 0.5,
        "eta_0": 0.1,
        "decay_delta": 0.25,
        "decay_eta": 0.75,
        "min_delta": 0.001,
        "min_eta": 0.001,
        "max_grad_norm": 5.0,
        "project_fn": {
            "_target_": "lama_lab.projectors.MarketMakingProjector",
            "low": 0.0,
            "high": 1.0,
            "epsilon": 0.001,
        },
    },
}

# Run the pipeline with custom overrides
run_pipeline(config=config, results_dir="../results")

Alternatively, invoke the script directly using one of the configurations available in `configs`.

In [ ]:
# Execute the script
!python ../scripts/market_making.py -c ../configs/market_making/pzomd_3fp.yml --results_dir ../results

## Save Results

The following cell is used to download the results as a zip archive from Google Colab.

In [ ]:
# Do NOT run this cell in local environment - it's intended for Google Colab only.

import shutil
from google.colab import files

# Compress the results folder into a ZIP archive
shutil.make_archive('../results', 'zip', '../results')

# Download the ZIP archive
files.download('../results.zip')

## Load Results

Retrieve and load artifacts from the latest experiment run.

In [ ]:
from lama_lab.utils import ResultsManager

manager = ResultsManager("../results")

# Retrieve the latest experiment
latest_exp_name = manager.list_experiments()[-1]
exp = manager.get_experiment(latest_exp_name)

# Load saved artifacts
artifacts = exp.load_all()

print(f"Successfully loaded experiment: {latest_exp_name}")
print("Available keys in artifacts:", list(artifacts.keys()))

## Interactive Visualization

Inspect market maker actions interactively.

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import lama_lab.plotting as plotting

In [ ]:
# Extract metrics and setup reference prices
metrics = artifacts["metrics"]

In [ ]:
# Plot actions. Narrow the slice to zoom in on the start of training, keeping in
# mind that a small learning rate can leave the first few thousand rounds flat.
selected_rounds = slice(None)

fig = plotting.plot_history(
    2,
    artifacts["actions"]["mean"][selected_rounds],
    artifacts["actions"]["min"][selected_rounds],
    artifacts["actions"]["max"][selected_rounds],
    artifacts["actions"]["std"][selected_rounds],
    feature_names=["Bid", "Ask"],
    feature_colors=["tab:blue", "tab:orange"],
    ylabel="Price",
    title_prefix="Action History",
    figsize=(12, 6)
)

plt.show()

## Learning Diagnostics

Inspect whether the empirical play of the two independent Exp3 learners is approaching a Nash equilibrium.

Exp3 has vanishing external regret, so the empirical distribution of play approaches the set of coarse correlated equilibria. That alone does not imply Nash: a time average of product distributions need not itself be a product distribution. Since a coarse correlated equilibrium that *is* a product distribution is exactly a mixed Nash equilibrium, both halves have to be checked, namely that the average regret vanishes **and** that the average joint distribution becomes close to the product of its marginals.

Every episode is an independent replica with its own weights, so the episodes act as independent seeds and each curve below is reported as the median with its interquartile range.

The last figure puts the three quantities on a single axis. They do not share units, the regret and the exploitability being expressed in payoff units and the independence defect being a total variation distance, but each is a gap that has to vanish for the claim to hold, which is what the comparison is about.

Requires an experiment run with a `diagnostics` block in its configuration, such as `configs/market_making/exp3_1fp.yml`.

In [ ]:
import numpy as np
import torch

from lama_lab.diagnostics import Exp3Diagnostics

if "diagnostics" not in artifacts:
    raise KeyError(
        "This experiment carries no diagnostics. Re-run with a configuration "
        "declaring a 'diagnostics' block, such as exp3_1fp.yml."
    )

# Recorded metrics, of shape (n_checkpoints, n_metrics, n_episodes)
table = artifacts["diagnostics"]["table"]
rounds = artifacts["diagnostics"]["checkpoints"].numpy()

METRIC_NAMES = Exp3Diagnostics.METRIC_NAMES
QUANTILES = torch.tensor([0.25, 0.5, 0.75])


def quantiles(name):
    """Return the lower quartile, the median and the upper quartile of a metric.

    Parameters
    ----------
    name : str
        One of ``Exp3Diagnostics.METRIC_NAMES``.

    Returns
    -------
    values : numpy.ndarray
        Array of shape ``(3, n_checkpoints)``, taken across the independent
        episodes, which play the role of independent seeds.
    """
    values = table[:, METRIC_NAMES.index(name)]
    return torch.nanquantile(values, QUANTILES, dim=-1).numpy()


# Table view of the figures below, and the readable form of the four criteria
reported = (
    "max_avg_regret",
    "independence_tv",
    "max_exploitability",
    "bound_1",
    "policy_drift_1",
    "policy_drift_2",
    "realized_vs_expected_l1",
)
summary = {name: quantiles(name) for name in reported}

print(f"{'T':>8}" + "".join(f"{name:>25}" for name in reported))
for step, n_round in enumerate(rounds):
    row = "".join(f"{summary[name][1][step]:>25.6f}" for name in reported)
    print(f"{n_round:>8}{row}")


In [ ]:
# Hues are kept apart under colour vision deficiency, unlike the usual
# blue/orange/green triple, whose green and orange are indistinguishable
PRIMARY, SECONDARY, TERTIARY = "#1f77b4", "#ff7f0e", "#9467bd"
REFERENCE, INK = "#8a8a8a", "#3a3a3a"


def draw(ax, name, color, label, marker="o", band=True):
    """Plot the median of a metric, optionally with its interquartile range."""
    lower, median, upper = quantiles(name)

    if band:
        ax.fill_between(rounds, lower, upper, color=color, alpha=0.15, linewidth=0)
    ax.plot(
        rounds,
        median,
        color=color,
        linewidth=2,
        marker=marker,
        markersize=6,
        label=label,
    )
    return median


def style(ax, title, ylabel):
    ax.set_xscale("log")
    ax.set_xlabel("Round")
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend(fontsize=8, frameon=False)


fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")

# Vanishing regret is the coarse correlated equilibrium half of the criterion
draw(axes[0, 0], "max_avg_regret", PRIMARY, "Expected policy")
draw(axes[0, 0], "avg_regret_realized_1", SECONDARY, "Realized play")
style(axes[0, 0], "Average external regret", "Regret per round")

# A vanishing defect is the independence half, and the non-trivial one
draw(axes[0, 1], "independence_tv", PRIMARY, "Time-averaged play")
axes[0, 1].axhline(
    0.0, color=REFERENCE, linewidth=1, linestyle="-.", label="Exact independence"
)
style(axes[0, 1], "Independence defect", "TV(joint, product of marginals)")

# Both halves together bound the distance from a Nash equilibrium. The bound
# holds per episode, so medians are compared with medians and no band is drawn
exploitability = draw(
    axes[1, 0], "max_exploitability", PRIMARY, "Exploitability", band=False
)
bound = summary["bound_1"][1]
axes[1, 0].plot(
    rounds,
    bound,
    color=REFERENCE,
    linewidth=2,
    linestyle="--",
    label="Regret + range x TV",
)
axes[1, 0].fill_between(
    rounds, exploitability, bound, color=REFERENCE, alpha=0.12, linewidth=0
)
style(axes[1, 0], "Nash exploitability of the product", "Best-response gain")

# Settled policies are stronger evidence than a vanishing defect alone
draw(axes[1, 1], "policy_drift_1", PRIMARY, "Maker 0")
draw(axes[1, 1], "policy_drift_2", SECONDARY, "Maker 1")
style(axes[1, 1], "Policy drift between checkpoints", "L1 change of the mean policy")

fig.suptitle("Exp3 convergence diagnostics (median and interquartile range)")
plt.show()

# The bound has to hold for every episode at every checkpoint
slack = table[:, METRIC_NAMES.index("bound_1")] - table[
    :, METRIC_NAMES.index("exploitability_1")
]
print(f"Smallest slack between the exploitability and its bound: {slack.min():+.3e}")


In [ ]:
# The three quantities share an axis on purpose: each is a gap that has to
# vanish for the empirical play to be approaching a Nash equilibrium
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")

series = (
    ("max_avg_regret", "Max average regret", PRIMARY, "o"),
    ("independence_tv", "Independence defect", SECONDARY, "s"),
    ("max_exploitability", "Max exploitability", TERTIARY, "^"),
)
endpoints = [
    (draw(ax, name, color, label, marker=marker)[-1], label)
    for name, label, color, marker in series
]

# Curves may end close together, so the labels are pushed apart in log space
previous = None
for value, label in sorted(endpoints):
    height = np.log10(value)
    if previous is not None:
        height = max(height, previous + 0.15)
    previous = height

    ax.annotate(
        label,
        (rounds[-1], 10.0**height),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=INK,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(rounds[0] * 0.8, rounds[-1] * 4.5)
ax.set_xlabel("Round")
ax.set_ylabel("Gap")
ax.set_title("Is the empirical play approaching a Nash equilibrium?", fontsize=11)
ax.grid(True, which="major", linestyle="--", alpha=0.7)
ax.grid(True, which="minor", linestyle="--", alpha=0.25)
ax.legend(fontsize=9, frameon=False, loc="lower left")

plt.show()

# Regret goes to zero by construction, so the reading rests on the other two:
#   defect and exploitability both fall   -> play approaches a Nash equilibrium
#   defect stays away from zero           -> a genuinely correlated equilibrium
#   defect falls, exploitability does not -> regret has not converged, or a bug


### Episode tail

The median can look converged while a large minority of episodes is stuck. Group the episodes by the exploitability of their **final** policy, not of the time average, and compare what each group settled on.

A healthy group has a small support only if it is sitting on a pure Nash profile. A small support with high exploitability means the policy collapsed onto the wrong arm and can no longer escape.

In [ ]:
final = torch.stack(
    [
        artifacts["diagnostics"]["final_policy_1"],
        artifacts["diagnostics"]["final_policy_2"],
    ]
)
arms = artifacts["trained_agents"][0].action_space.cpu()

# Pure Nash profiles of the grid game, as arm-index pairs
grid_nash = {
    tuple(p) for p in artifacts["metrics"]["finite_grid_pure_nash"]["indices"]
}
print(f"{len(grid_nash)} pure Nash profiles on the grid")

last = table[-1]
exploitability = last[METRIC_NAMES.index("max_last_exploitability")]
groups = {
    "good  (< 1e-3)": exploitability < 1e-3,
    "inter (< 1e-2)": (exploitability >= 1e-3) & (exploitability < 1e-2),
    "bad   (>= 1e-2)": exploitability >= 1e-2,
}

modal = final.argmax(dim=-1)
for label, mask in groups.items():
    count = int(mask.sum())
    print(f"\n{label}: {count} of {mask.numel()} episodes")
    if count == 0:
        continue

    for name in ("support_1", "nash_mass", "independence_tv"):
        value = last[METRIC_NAMES.index(name)][mask].median()
        print(f"    median {name:<16} {value:.4f}")

    profiles = {}
    for episode in mask.nonzero().flatten().tolist():
        key = (modal[0, episode].item(), modal[1, episode].item())
        profiles[key] = profiles.get(key, 0) + 1

    for (i, j), n in sorted(profiles.items(), key=lambda kv: -kv[1])[:4]:
        tag = "nash" if (i, j) in grid_nash else "----"
        print(f"    {n:>3}x  {tag}  ({arms[i].tolist()}, {arms[j].tolist()})")
